# Quilt on the Synthetic Dataset
### This Jupyter Notebook simulates Quilt method on the synthetic data.

## Import libraries

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from model import NormalNN, EarlyStopping, NNClassifier
from utils import prepare_data
from Quilt import Quilt_SFS, Quilt_SFFS, Quilt_GA

## Load data

In [2]:
x_all = np.load('./dataset/SEA_data.npy')
y_all = np.load('./dataset/SEA_label.npy')
concept_drifts = np.load('./dataset/SEA_drift_point.npy')

In [3]:
cuda = True if torch.cuda.is_available() else False

if cuda:
    torch.cuda.set_device(0)

## Quilt(SFS)

In [4]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):
    
    n_dataset = n+1
    n_feature = x_all.shape[1]

    # ---------------------
    #  Define Quilt
    # ---------------------
    quilt = Quilt_SFS(x_all, y_all, n_dataset, n_feature, concept_drifts, num)
    
    # ---------------------
    #  Data calibration and sampling
    # ---------------------
    scaler_list = [MinMaxScaler(), StandardScaler(), RobustScaler()]
    x_all_cali, y_all_cali = quilt.calibration(n, scaler_list)
    x_sample, y_sample, concept_drifts_sample = quilt.sampling(n, x_all_cali, y_all_cali)
    
    print("test segment: ", n_dataset)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):

        # ---------------------
        #  Feature Selection and Calibration(FSC) + Data Segment Selection(DS)
        # ---------------------
        chromo_df_pcos,score_pcos = quilt.generation(n, x_sample, y_sample, concept_drifts_sample, 
                                                         lr=lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Selected and calibrated (data segment, feature) result
        # ---------------------
        dataset_final = []
        feature_final = []
        
        for i in range(n_dataset):
            if chromo_df_pcos[i]:
                dataset_final.append(i)
                
        for j in range(n_dataset, n_dataset+n_feature*(1+len(scaler_list))):
            if chromo_df_pcos[j]:
                feature_final.append(j-n_dataset)
                
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        # ---------------------
        #  Construct new train data with Quilt result 
        # ---------------------
        train_ds, valid_ds, test_ds = prepare_data(n, x_all_cali, y_all_cali, concept_drifts, 
                                                   dataset_final, feature_final)
        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=len(feature_final), seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, 
                earlystop_path='checkpoint_SEA_Quilt_SFS.pt')
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  7
test acc: avg 0.901889, std 0.012681
data usage(#feature, #segment, %total): (3.000000, 3.000000, 42.857143%)
----------------------------------------------------------------------------
test segment:  8
test acc: avg 0.894778, std 0.001030
data usage(#feature, #segment, %total): (2.000000, 3.800000, 31.666667%)
----------------------------------------------------------------------------
test segment:  9
test acc: avg 0.904000, std 0.004745
data usage(#feature, #segment, %total): (2.200000, 4.000000, 32.592593%)
----------------------------------------------------------------------------
overall test acc: avg 0.900222, std 0.006152
overall data usage(#feature, #segment, %total): (2.400000, 3.600000, 35.705467%)
